# 01 - Extracción de Datos

**Objetivo:** Extraer datos de WideWorldImporters y crear tabla RAW en WideWorldImportersDW2

**Salida:**
- Tabla: `raw.invoice_lines`
- Archivos: `datos/bronze/raw_invoice_lines_LOTE_*.csv`

In [ ]:
import pandas as pd
import json
import hashlib
from pathlib import Path
from datetime import datetime
import sys
import uuid

# Agregar el módulo de conexión al path
sys.path.insert(0, str(Path.cwd().parent))
from conexion_sqlserver import obtener_engines

print("✓ Librerías importadas")

## 1. Obtener conexiones a BD

In [ ]:
# Obtener engines
engine_origen, engine_destino = obtener_engines()

print("✓ Conectado a SQL Server")
print(f"  - Origen: WideWorldImporters")
print(f"  - Destino: WideWorldImportersDW2")

## 2. Crear tabla RAW en destino

In [ ]:
# Crear schema y tabla RAW
with engine_destino.connect() as conn:
    # Crear schema si no existe
    conn.execute("""
        IF NOT EXISTS (SELECT * FROM sys.schemas WHERE name = 'raw')
        BEGIN
            EXEC sp_executesql N'CREATE SCHEMA raw'
        END
    """)
    
    # Crear tabla RAW
    conn.execute("""
        IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'invoice_lines' AND SCHEMA_NAME(schema_id) = 'raw')
        BEGIN
            CREATE TABLE raw.invoice_lines (
                id INT PRIMARY KEY IDENTITY(1,1),
                lote_id VARCHAR(36) NOT NULL,
                tabla_origen VARCHAR(100) NOT NULL,
                numero_fila INT NOT NULL,
                payload NVARCHAR(MAX) NOT NULL,
                hash_sha256 VARCHAR(64) NOT NULL,
                fecha_carga DATETIME NOT NULL DEFAULT GETDATE()
            )
        END
    """)
    
    conn.commit()

print("✓ Tabla raw.invoice_lines creada/verificada")

## 3. Extraer datos del origen

In [ ]:
# Extraer datos
query = """SELECT TOP 100 * FROM Sales.InvoiceLines"""
df = pd.read_sql_query(query, engine_origen)

print(f"✓ Extraídos {len(df)} registros")
print(f"  Columnas: {list(df.columns)}")

## 4. Procesar datos y crear payloads

In [ ]:
# Generar ID de lote
lote_id = str(uuid.uuid4())
tabla_origen = "Sales.InvoiceLines"

# Procesar registros
registros_raw = []

for idx, row in df.iterrows():
    # Crear payload JSON
    row_dict = row.to_dict()
    
    # Convertir valores no serializables
    for key, value in row_dict.items():
        if pd.isna(value):
            row_dict[key] = None
        elif isinstance(value, (pd.Timestamp, datetime)):
            row_dict[key] = value.isoformat()
    
    payload_json = json.dumps(row_dict, default=str)
    
    # Calcular hash
    hash_sha256 = hashlib.sha256(payload_json.encode()).hexdigest()
    
    registros_raw.append({
        "lote_id": lote_id,
        "tabla_origen": tabla_origen,
        "numero_fila": idx + 1,
        "payload": payload_json,
        "hash_sha256": hash_sha256
    })

print(f"✓ Procesados {len(registros_raw)} registros con payloads y hashes")
print(f"  Lote ID: {lote_id}")

## 5. Guardar en BD

In [ ]:
# Convertir a DataFrame
df_raw = pd.DataFrame(registros_raw)

# Insertar en BD
df_raw.to_sql('invoice_lines', engine_destino, schema='raw', if_exists='append', index=False)

print(f"✓ Insertados {len(df_raw)} registros en raw.invoice_lines")

## 6. Guardar en archivos CSV

In [ ]:
# Crear carpeta de datos
datos_dir = Path.cwd() / "datos" / "bronze"
datos_dir.mkdir(parents=True, exist_ok=True)

# Guardar archivo CSV
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = datos_dir / f"raw_invoice_lines_LOTE_{lote_id[:8]}.csv"

df_raw.to_csv(csv_path, index=False, encoding='utf-8')

print(f"✓ Archivo guardado: {csv_path}")
print(f"  Tamaño: {csv_path.stat().st_size / 1024:.1f} KB")

## Resumen

In [ ]:
print("\n" + "="*60)
print("RESUMEN - ETAPA 01: EXTRACCION")
print("="*60)
print(f"✓ Registros extraídos: {len(registros_raw)}")
print(f"✓ Tabla BD: raw.invoice_lines")
print(f"✓ Archivo CSV: {csv_path.name}")
print(f"✓ Lote ID: {lote_id}")
print("="*60)